# Tuần 11: MT evaluation

Mục tiêu: dùng `sacrebleu` để tính BLEU, chrF++ và TER cho bản dịch máy Hán-Việt, rồi đọc kết quả cùng human error labels.

## Trước khi chạy code

Cell setup là **run-only**: import thư viện, kiểm tra data và tạo output folders.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import sys
import subprocess

try:
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sacrebleu.metrics import BLEU, CHRF, TER
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "matplotlib", "sacrebleu"])
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sacrebleu.metrics import BLEU, CHRF, TER

CANDIDATES = [Path("weeks/week-11-mt-evaluation"), Path(".")]
WEEK_DIR = next(
    candidate for candidate in CANDIDATES
    if (candidate / "data" / "raw" / "week11_mt_evaluation_segments.csv").exists()
    or candidate.name == "week-11-mt-evaluation"
)
DATA_PATH = WEEK_DIR / "data" / "raw" / "week11_mt_evaluation_segments.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA = "b86c4ea84fcccba42c1a6dd8e6ac4112e42e59198ceb4b6c9e98dea0c810a939"
REMOTE_DATA = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-11-mt-evaluation/data/raw/week11_mt_evaluation_segments.csv"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(REMOTE_DATA, DATA_PATH)
actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)
if actual_sha != EXPECTED_SHA:
    print("Note: SHA differs from the course snapshot. Continue only if you intentionally changed the data.")


Data file: weeks/week-11-mt-evaluation/data/raw/week11_mt_evaluation_segments.csv
SHA-256: b86c4ea84fcccba42c1a6dd8e6ac4112e42e59198ceb4b6c9e98dea0c810a939


## 1. Load MT evaluation data

Một row là một source segment được dịch bởi một MT system.

In [2]:
data = pd.read_csv(DATA_PATH)
print("Rows:", len(data))
print("Source segments:", data["segment_id"].nunique())
print("Systems:", ", ".join(sorted(data["mt_system"].unique())))
print(data[["segment_id", "mt_system", "zh_source", "vi_mt_output", "vi_reference"]].head(6).to_string(index=False))


Rows: 36
Source segments: 12
Systems: MT_A_literal, MT_B_fluent, MT_C_omission
segment_id     mt_system            zh_source                                                                                  vi_mt_output                                                                      vi_reference
      S001  MT_A_literal 推进教育数字化有助于扩大优质资源覆盖面。                   Thúc đẩy giáo dục số hóa có ích cho mở rộng mặt bao phủ tài nguyên ưu chất.    Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.
      S001   MT_B_fluent 推进教育数字化有助于扩大优质资源覆盖面。 Thúc đẩy chuyển đổi số trong giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao.    Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.
      S001 MT_C_omission 推进教育数字化有助于扩大优质资源覆盖面。                                        Số hóa giáo dục giúp mở rộng nguồn lực chất lượng cao.    Thúc đẩy số hóa giáo dục giúp mở rộng độ bao phủ của nguồn lực chất lượng cao.
      S002  MT_A_literal  教师培训应更加

## 2. Compute metrics by system

BLEU và chrF++: cao hơn thường tốt hơn. TER: thấp hơn thường tốt hơn.

In [3]:
bleu = BLEU(effective_order=True)
chrf = CHRF(word_order=2)
ter = TER()

metric_rows = []
for system, rows in data.groupby("mt_system"):
    hypotheses = rows["vi_mt_output"].tolist()
    references = [rows["vi_reference"].tolist()]
    metric_rows.append({
        "mt_system": system,
        "segments": len(rows),
        "bleu": round(bleu.corpus_score(hypotheses, references).score, 2),
        "chrf_pp": round(chrf.corpus_score(hypotheses, references).score, 2),
        "ter": round(ter.corpus_score(hypotheses, references).score, 2),
        "mean_adequacy": round(rows["human_adequacy_1_5"].mean(), 2),
        "mean_fluency": round(rows["human_fluency_1_5"].mean(), 2),
        "major_or_minor_errors": int(rows["severity"].isin(["major", "minor"]).sum()),
    })
metric_summary = pd.DataFrame(metric_rows).sort_values(["chrf_pp", "bleu"], ascending=False)
metric_summary.to_csv(TABLE_DIR / "week11_system_metric_summary.csv", index=False)
print(metric_summary.to_string(index=False))


    mt_system  segments  bleu  chrf_pp   ter  mean_adequacy  mean_fluency  major_or_minor_errors
  MT_B_fluent        12 63.47    77.94 23.04           4.39          4.19                     10
 MT_A_literal        12 34.72    55.82 47.06           2.55          2.25                     12
MT_C_omission        12 32.96    53.29 49.02           3.08          3.28                     12


## 3. Human error label summary

Metric cho ta một số; human label cho ta lý do cần sửa.

In [4]:
error_summary = (
    data.groupby(["mt_system", "simplified_error_label"], as_index=False)
    .size()
    .rename(columns={"size": "segment_count"})
    .sort_values(["mt_system", "segment_count"], ascending=[True, False])
)
error_summary.to_csv(TABLE_DIR / "week11_error_label_summary.csv", index=False)
print(error_summary.to_string(index=False))


    mt_system    simplified_error_label  segment_count
 MT_A_literal terminology_or_word_order             12
  MT_B_fluent               minor_style             10
  MT_B_fluent        acceptable_variant              2
MT_C_omission                  omission             12


## 4. Segment review sample

Đọc vài rows để nhớ metric không tự giải thích lỗi dịch.

In [5]:
review_sample = data[data["segment_id"].isin(["S001", "S008", "S012"])].copy()
review_sample.to_csv(TABLE_DIR / "week11_segment_review_sample.csv", index=False)
print(review_sample[["segment_id", "mt_system", "vi_mt_output", "simplified_error_label", "severity"]].to_string(index=False))


segment_id     mt_system                                                                                  vi_mt_output    simplified_error_label severity
      S001  MT_A_literal                   Thúc đẩy giáo dục số hóa có ích cho mở rộng mặt bao phủ tài nguyên ưu chất. terminology_or_word_order    major
      S001   MT_B_fluent Thúc đẩy chuyển đổi số trong giáo dục giúp mở rộng phạm vi tiếp cận nguồn lực chất lượng cao.               minor_style    minor
      S001 MT_C_omission                                        Số hóa giáo dục giúp mở rộng nguồn lực chất lượng cao.                  omission    major
      S008  MT_A_literal        Khóa Hán ngữ ngắn hạn nên kết hợp bối cảnh mẫu ngữ của người học để thiết kế nhiệm vụ. terminology_or_word_order    major
      S008   MT_B_fluent       Khóa Hán ngữ ngắn hạn nên thiết kế nhiệm vụ gắn với nền tảng tiếng mẹ đẻ của người học.               minor_style    minor
      S008 MT_C_omission                                    Khóa Hán ngữ ngắ

## 5. Figures cho paper

Figure 1 là core. Figure 2 là stretch nếu còn thời gian.

In [6]:
plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 11})
plot = metric_summary.sort_values("chrf_pp")
fig, ax = plt.subplots(figsize=(7.8, 4.8))
ax.barh(plot["mt_system"], plot["chrf_pp"], color="#2f63ea")
ax.set_title("Week 11 MT evaluation: chrF++ by system")
ax.set_xlabel("chrF++ (higher is better)")
ax.set_ylabel("MT system")
for i, value in enumerate(plot["chrf_pp"]):
    ax.text(value + 0.4, i, f"{value:.1f}", va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week11_chrf_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week11_chrf_by_system.svg")
plt.close(fig)

error_counts = data[data["severity"].isin(["major", "minor"])].groupby("mt_system").size().reindex(metric_summary["mt_system"]).fillna(0)
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.bar(error_counts.index, error_counts.values, color="#b45309")
ax.set_title("Week 11 simplified human error count by system")
ax.set_xlabel("MT system")
ax.set_ylabel("Segments with simplified error label")
for i, value in enumerate(error_counts.values):
    ax.text(i, value + 0.2, str(int(value)), ha="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week11_error_count_by_system.png", dpi=200)
fig.savefig(FIG_DIR / "week11_error_count_by_system.svg")
plt.close(fig)
print("Figures exported:")
print(FIG_DIR / "week11_chrf_by_system.png")
print(FIG_DIR / "week11_error_count_by_system.png")


Figures exported:
weeks/week-11-mt-evaluation/outputs/figures/week11_chrf_by_system.png
weeks/week-11-mt-evaluation/outputs/figures/week11_error_count_by_system.png


## 6. Results paragraph

Điền số liệu từ metric table. Luôn nói rõ limitation: one reference, synthetic data, human review needed.

In [7]:
best_chrf = metric_summary.iloc[0]
best_ter = metric_summary.sort_values("ter").iloc[0]
max_error_count = metric_summary["major_or_minor_errors"].max()
error_tie = metric_summary.loc[metric_summary["major_or_minor_errors"] == max_error_count, "mt_system"]
error_list = list(error_tie)
if len(error_list) == 1:
    error_systems = error_list[0]
else:
    error_systems = ", ".join(error_list[:-1]) + " and " + error_list[-1]
paragraph = (
    f"In the synthetic Week 11 MT evaluation dataset, {best_chrf['mt_system']} had the highest chrF++ "
    f"score ({best_chrf['chrf_pp']}) across {int(best_chrf['segments'])} segments, while "
    f"{best_ter['mt_system']} had the lowest TER ({best_ter['ter']}). Human review showed that "
    f"{error_systems} had the highest simplified error-label count ({int(max_error_count)} rows). "
    "This suggests that automatic metrics are useful for comparing systems, but they should be read "
    "with segment-level error notes. Because the dataset is synthetic and uses one Vietnamese reference "
    "per source segment, the result should not be reported as a general claim about MT quality."
)
print(paragraph)


In the synthetic Week 11 MT evaluation dataset, MT_B_fluent had the highest chrF++ score (77.94) across 12 segments, while MT_B_fluent had the lowest TER (23.04). Human review showed that MT_A_literal and MT_C_omission had the highest simplified error-label count (12 rows). This suggests that automatic metrics are useful for comparing systems, but they should be read with segment-level error notes. Because the dataset is synthetic and uses one Vietnamese reference per source segment, the result should not be reported as a general claim about MT quality.
